# PCA & Dimensionality Reduction

1. **PCA** - Principal Component Analysis, explained variance
2. **t-SNE** - Nonlinear embedding for visualization
3. **PCA for preprocessing** - Reducing dimensions before classification

**Dataset**: Digits (64 dimensions) and Breast Cancer

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_digits, load_breast_cancer
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LogisticRegression

sns.set_theme(style="whitegrid")

In [ ]:
digits = load_digits()
X, y = digits.data, digits.target
X_scaled = StandardScaler().fit_transform(X)

print(f"Original shape: {X.shape} (64 pixel features)")

In [ ]:
# PCA: how much variance does each component explain?
pca_full = PCA().fit(X_scaled)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(range(1, len(pca_full.explained_variance_ratio_) + 1),
            pca_full.explained_variance_ratio_, color="teal", alpha=0.7)
axes[0].set_xlabel("Principal Component")
axes[0].set_ylabel("Explained Variance Ratio")
axes[0].set_title("Variance per Component")

cumulative = np.cumsum(pca_full.explained_variance_ratio_)
axes[1].plot(range(1, len(cumulative) + 1), cumulative, "o-", color="teal")
axes[1].axhline(y=0.95, color="coral", linestyle="--", label="95% variance")
n_95 = np.argmax(cumulative >= 0.95) + 1
axes[1].axvline(x=n_95, color="coral", linestyle="--", alpha=0.5)
axes[1].set_xlabel("Number of Components")
axes[1].set_ylabel("Cumulative Explained Variance")
axes[1].set_title(f"95% variance with {n_95} components (out of {X.shape[1]})")
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# 2D PCA projection
X_pca_2d = PCA(n_components=2).fit_transform(X_scaled)

plt.figure(figsize=(8, 6))
scatter = plt.scatter(X_pca_2d[:, 0], X_pca_2d[:, 1], c=y, cmap="tab10", s=10, alpha=0.7)
plt.colorbar(scatter, label="Digit")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("PCA: Digits projected to 2D")
plt.tight_layout()
plt.show()

## t-SNE: Nonlinear Visualization

t-SNE preserves local structure better than PCA for visualization.

In [ ]:
# t-SNE (use PCA first to speed up)
X_pca_30 = PCA(n_components=30).fit_transform(X_scaled)
X_tsne = TSNE(n_components=2, perplexity=30, random_state=42).fit_transform(X_pca_30)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

axes[0].scatter(X_pca_2d[:, 0], X_pca_2d[:, 1], c=y, cmap="tab10", s=10, alpha=0.7)
axes[0].set_title("PCA (2D)")

axes[1].scatter(X_tsne[:, 0], X_tsne[:, 1], c=y, cmap="tab10", s=10, alpha=0.7)
axes[1].set_title("t-SNE (2D)")

plt.suptitle("PCA vs t-SNE: Digits Dataset")
plt.tight_layout()
plt.show()

## PCA as Preprocessing: Effect on Classification

In [ ]:
components_range = [5, 10, 20, 30, 40, 50, 64]
scores = []

for n in components_range:
    pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("pca", PCA(n_components=n)),
        ("clf", LogisticRegression(max_iter=5000, random_state=42)),
    ])
    cv = cross_val_score(pipe, X, y, cv=5, scoring="accuracy")
    scores.append(cv.mean())
    print(f"n_components={n:3d}: accuracy={cv.mean():.4f} (+/- {cv.std():.4f})")

plt.figure(figsize=(8, 5))
plt.plot(components_range, scores, "o-", color="teal")
plt.xlabel("Number of PCA Components")
plt.ylabel("CV Accuracy")
plt.title("Classification Accuracy vs PCA Dimensions")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Key Takeaways

1. **PCA is linear** - preserves global structure, fast, deterministic
2. **t-SNE is nonlinear** - great for visualization, but slow and non-deterministic
3. **Choose components by explained variance** - 95% is a common threshold
4. **PCA as preprocessing can improve speed and performance** by removing noise dimensions
5. **t-SNE should only be used for visualization** - not as a preprocessing step for ML models